# NASA CMAPSS — Feature Engineering
## Remove constant/low-variance sensors from train.csv and test.csv

This notebook:
1. Loads `data/raw/train.csv` and `data/raw/test.csv` produced by notebook 01_data_preprocessing_RUL
2. Identifies constant or near-constant sensors per dataset subset (FD001–FD004)
3. Drops sensors that carry no degradation signal (variance below threshold)
4. Reports which sensors were dropped and why
5. Saves `data/processed/train.csv` and `data/processed/test.csv`

In [ ]:
import pandas as pd
import numpy as np
import os

# Variance threshold — sensors with std below this are considered constant
# Using 0.1 as a conservative threshold; adjust if needed
STD_THRESHOLD = 0.1

# Sensors known to be constant in the literature (for reference)
KNOWN_CONSTANT_SENSORS = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']

SENSOR_COLS = [f's{i}' for i in range(1, 22)]

os.makedirs('../data/processed', exist_ok=True)

## Load Data

In [ ]:
train = pd.read_csv('../data/raw/train.csv')
test  = pd.read_csv('../data/raw/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"\nColumns: {list(train.columns)}")
train.head()

## Step 1: Compute Per-Sensor Standard Deviation (Train Only)

We evaluate sensor variance **on the training set only** to avoid data leakage.
The same dropped columns will be applied to the test set.

We check variance **per dataset_id** because FD002/FD004 have 6 operating conditions —
a sensor that looks variable globally might only vary due to operating condition shifts,
not actual degradation. A sensor must show variation within at least one subset to be kept.

In [ ]:
# Global std across all data
global_std = train[SENSOR_COLS].std().sort_values()

print("=== Global Sensor Standard Deviations (ascending) ===")
print(global_std.to_string())

print(f"\nSensors with std < {STD_THRESHOLD} (likely constant):")
print(global_std[global_std < STD_THRESHOLD].index.tolist())

## Step 2: Per-Dataset-ID Variance Check

Check variance within each FD00X subset independently.

In [ ]:
print("=== Per-Dataset Standard Deviations ===")
per_dataset_std = train.groupby('dataset_id')[SENSOR_COLS].std()
print(per_dataset_std.T.to_string())

# A sensor is 'constant' in a dataset if its std < threshold in that dataset
# A sensor is globally droppable if it is constant in ALL datasets
is_constant_per_dataset = per_dataset_std < STD_THRESHOLD  # shape: (4, 21)
constant_in_all = is_constant_per_dataset.all(axis=0)       # True if constant in every dataset

print(f"\nSensors constant across ALL datasets (std < {STD_THRESHOLD} in every subset):")
print(constant_in_all[constant_in_all].index.tolist())

## Step 3: Decide Which Sensors to Drop

We combine two signals:
- **Data-driven**: sensors with global std < threshold
- **Literature-confirmed**: known constant sensors from the CMAPSS paper

The union of both lists is dropped.

In [ ]:
# Data-driven: constant in ALL dataset subsets
data_driven_drops = constant_in_all[constant_in_all].index.tolist()

# Union with literature-known constants
sensors_to_drop = sorted(set(data_driven_drops)) 
sensors_to_keep = [s for s in SENSOR_COLS if s not in sensors_to_drop]

print(f"Sensors to DROP ({len(sensors_to_drop)}): {sensors_to_drop}")
print(f"Sensors to KEEP ({len(sensors_to_keep)}): {sensors_to_keep}")

# Cross-check: confirm no sensor in 'to keep' has suspiciously low global variance
kept_std = global_std[sensors_to_keep]
print(f"\nStd of kept sensors (min should be well above {STD_THRESHOLD}):")
print(kept_std.sort_values().to_string())

## Step 4: Drop Constant Sensors

In [ ]:
# Columns to retain: non-sensor columns + kept sensors
non_sensor_cols = ['engine_id', 'cycle', 'op1', 'op2', 'op3', 'dataset_id', 'RUL']
retain_cols = non_sensor_cols + sensors_to_keep

train_clean = train[retain_cols].copy()
test_clean  = test[retain_cols].copy()

print(f"Train: {train.shape} → {train_clean.shape}")
print(f"Test:  {test.shape}  → {test_clean.shape}")
print(f"\nDropped {len(sensors_to_drop)} sensor columns: {sensors_to_drop}")
print(f"Remaining columns: {list(train_clean.columns)}")
train_clean.head()

## Save to CSV

In [ ]:
train_clean.to_csv('../data/processed/train.csv', index=False)
test_clean.to_csv('../data/processed/test.csv', index=False)
print("Saved: ../data/processed/train.csv and ../data/processed/test.csv")

## Verification

In [ ]:
print("=== VERIFICATION ===\n")

# 1. No dropped sensor should appear in the output
for s in sensors_to_drop:
    assert s not in train_clean.columns, f"ERROR: {s} still present in train!"
    assert s not in test_clean.columns,  f"ERROR: {s} still present in test!"
print(f"All {len(sensors_to_drop)} dropped sensors confirmed absent")

# 2. Row counts must be unchanged
assert len(train_clean) == len(train), "ERROR: Train row count changed!"
assert len(test_clean)  == len(test),  "ERROR: Test row count changed!"
print(f"Row counts unchanged — Train: {len(train_clean)}, Test: {len(test_clean)}")

# 3. Engine counts must be unchanged
assert train_clean['engine_id'].nunique() == train['engine_id'].nunique()
assert test_clean['engine_id'].nunique()  == test['engine_id'].nunique()
print(f"Engine counts unchanged — Train: {train_clean['engine_id'].nunique()}, "
      f"Test: {test_clean['engine_id'].nunique()}")

# 4. RUL values must not have changed
assert train_clean['RUL'].equals(train['RUL']), "ERROR: RUL values changed in train!"
assert test_clean['RUL'].equals(test['RUL']),   "ERROR: RUL values changed in test!"
print(f"RUL values unchanged")

# 5. Remaining sensors must all have std > threshold
remaining_sensors = [c for c in train_clean.columns if c.startswith('s')]
low_var = train_clean[remaining_sensors].std()
low_var_sensors = low_var[low_var < STD_THRESHOLD].index.tolist()
if low_var_sensors:
    print(f"WARNING: These kept sensors have low variance: {low_var_sensors}")
else:
    print(f"All {len(remaining_sensors)} remaining sensors have std >= {STD_THRESHOLD}")

print(f"\nFinal dataset summary:")
print(f"  Sensors kept: {remaining_sensors}")
print(f"  Sensors dropped: {sensors_to_drop}")
print(f"  Train shape: {train_clean.shape}")
print(f"  Test shape:  {test_clean.shape}")

In [ ]:
print("\nFeature engineering (constant sensor removal) complete!")